# 02 · Trajectory models, evidence, and decisions

**NFL Player Trajectory Lab** · Frozen temporal validation · Official coordinate RMSE

**Current measured winner: landing-aware residual ridge, 0.9269 RMSE.** This notebook moves from physical references to learned motion, feature ablations, error budgets, and an explicit next-experiment decision. The 48-game holdout stays reserved. Results are local temporal validation, not a Kaggle leaderboard claim.

The final section lets the project owner generate and download their own inference notebook. Export is off by default, and nothing is submitted automatically.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown(
        "Run `nfl benchmark` after the data audit to create the real-data results. "
        "No synthetic result is substituted here."
    ))

## The official metric

$$\mathrm{RMSE}=\sqrt{\frac{\sum_{i=1}^{N}[(\hat{x}_i-x_i)^2+(\hat{y}_i-y_i)^2]}{2N}}.$$

Coordinates are measured in yards. Every coordinate/frame receives equal weight.
ADE measures Euclidean displacement, FDE measures each trajectory's final displacement,
and p95 reports the error tail. They complement RMSE; they are different quantities.

In [ ]:
if READY:
    scores = pd.DataFrame(summary["models"])
    display(scores[["model", "coordinate_rmse_yards", "rmse_ci95_low", "rmse_ci95_high", "ade_frame_weighted_yards", "fde_trajectory_weighted_yards", "p95_displacement_yards"]].round(4))
    best = scores.iloc[0]
    display(Markdown(
        f"**Original learned baseline:** `{best['model']}`. "
        f"RMSE is **{summary['improvement_vs_velocity_percent']:.1f}% lower** "
        "than constant velocity on these validation games."
    ))

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "benchmark.png")))

## Interpret the learned weights

The six basis vectors are last velocity × time, recent velocity × time,
acceleration × time²/2, and the vector to the landing point multiplied by normalized
time, normalized time², and normalized time³. Training-RMS scaling and ridge stabilize
the fit. Coordinates share coefficients, preserving rotation and translation equivariance.
An unseen role uses the global training fit. Correlated weights should be interpreted jointly.

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "coefficients.png")))
    latency = json.loads((RESULTS / "latency.json").read_text())
    display(pd.DataFrame([latency]))

## Watch trajectories and reproduce the experiment

Open `artifacts/benchmark/report.html` for the full offline report and animated field.
The example is the first validation play by game/play ID, selected independently of its errors.

A repeat of `nfl benchmark` verifies cached files and reuses completed weekly stages.
Changed inputs, model source, or dependencies invalidate corresponding checkpoints.
Interrupted stages are recomputed, and a corrupt artifact never counts as completed.

## What this result establishes

Game-cluster bootstrap intervals resample entire games 2,000 times, preserving player
and frame dependence within each sampled game. Paired intervals compare against
constant velocity. They describe this validation sample, not every future season.
Repeated model selection can overfit validation. Keep the holdout reserved until
model selection is locked.

## Where the baseline still struggles

Read the role and forecast-time slices rather than only the overall average.
These are development-validation diagnostics, not new holdout results. Longer
forecasts and defensive coverage motivate the interaction feature experiment.

In [ ]:
if READY:
    slices = pd.DataFrame(summary["slices"])
    display(slices.loc[slices["model"].eq("role_ridge"), ["dimension", "value", "rows", "coordinate_rmse_yards"]].round(4))

## Completed feature ablations

All models below score the same 67,857 player-frames from 32 validation games. Scaling and feature screening use training games only. The comparison uses the official pooled coordinate RMSE. The paired intervals versus role ridge resample the same games; overlapping marginal intervals are not a valid test of a paired difference.

In [ ]:
import io

import matplotlib.pyplot as plt

from nfl_trajectory.research import load_evidence
from nfl_trajectory.runtime import Run

feature_summary, selection, evidence_label = load_evidence(ROOT)
display(Markdown(f"**Feature evidence:** {evidence_label}"))


def display_figure(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    plt.close(figure)
    display(Image(data=buffer.getvalue()))

In [ ]:
with Run(ROOT, "feature_result_analysis") as run:
    feature_scores = pd.DataFrame(feature_summary["models"])
    columns = ["model", "selected_features", "coordinate_rmse_yards",
               "ade_frame_weighted_yards", "fde_trajectory_weighted_yards",
               "p95_displacement_yards", "delta_vs_role_ridge_ci95"]
    readable = feature_scores[columns].copy()
    readable["delta_vs_role_ridge_ci95"] = readable["delta_vs_role_ridge_ci95"].map(
        lambda bounds: f"[{bounds[0]:.4f}, {bounds[1]:.4f}]"
    )
    display(readable.rename(columns={
        "model": "Model", "selected_features": "Features", "coordinate_rmse_yards": "RMSE (yd)",
        "ade_frame_weighted_yards": "ADE (yd)", "fde_trajectory_weighted_yards": "FDE (yd)",
        "p95_displacement_yards": "P95 (yd)", "delta_vs_role_ridge_ci95": "Paired ΔRMSE 95% CI",
    }).round(4))
    winner = feature_scores.iloc[0]
    reference = feature_scores.set_index("model").loc["role_ridge"]
    improvement = 100 * (1 - winner["coordinate_rmse_yards"] / reference["coordinate_rmse_yards"])
    paired_upper = winner["delta_vs_role_ridge_ci95"][1]
    uncertainty = (
        "The paired 95% interval versus role ridge is entirely below zero. "
        if paired_upper < 0 else "The paired 95% interval does not establish an improvement. "
    )
    display(Markdown(
        f"**Decision:** retain `{winner['model']}` as the development champion. "
        f"Its RMSE is **{improvement:.2f}% lower** than role ridge. "
        + uncertainty +
        "This supports improvement on these validation games, not a guarantee on the holdout."
    ))
    positions = list(range(len(feature_scores)))
    fig, ax = plt.subplots(figsize=(10, 4.7), layout="constrained")
    low = feature_scores["coordinate_rmse_yards"] - feature_scores["rmse_ci95_low"]
    high = feature_scores["rmse_ci95_high"] - feature_scores["coordinate_rmse_yards"]
    ax.errorbar(feature_scores["coordinate_rmse_yards"], positions,
                xerr=[low, high], fmt="o", capsize=4)
    ax.set_yticks(positions, feature_scores["model"].str.replace("_", " "))
    for position, value in enumerate(feature_scores["coordinate_rmse_yards"]):
        ax.annotate(f"{value:.4f}", (value, position), xytext=(0, 9),
                    textcoords="offset points", ha="center", fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Coordinate RMSE in yards; lower is better")
    ax.set_title("Real-data ablation | 95% game-cluster intervals", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    display_figure(fig)
    run.event("champion_reviewed", model=winner["model"], rmse=float(winner["coordinate_rmse_yards"]),
              holdout_evaluation="not_run")

## Where the remaining error is concentrated

The contribution of a slice to total squared coordinate error is proportional to `rows × RMSE²`. The factor of two coordinates cancels in the percentage. This respects the official metric; averaging slice RMSEs would not.

Forecast second 1 covers frames 1–10 (0.1–1.0 seconds); second 2 covers 1.1–2.0 seconds, and so on. The fourth-second slice has only 127 frames and should not dominate a research decision.

In [ ]:
from nfl_trajectory.research import error_budget

with Run(ROOT, "error_budget_analysis") as run:
    roles = error_budget(feature_summary, "role")
    horizons = error_budget(feature_summary, "forecast_second")
    display(roles.round(3))
    display(horizons.round(3))
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    ax.bar(horizons["value"], horizons["squared_error_share_percent"])
    ax.set_xlabel("Forecast second")
    ax.set_ylabel("Share of remaining squared error (%)")
    ax.set_title("Second- and third-second forecasts dominate the error budget", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    display_figure(fig)
    defense_share = roles.set_index("value").loc["Defensive Coverage", "squared_error_share_percent"]
    display(Markdown(
        f"**Priority:** defensive coverage accounts for **{defense_share:.1f}%** "
        "of squared error."
    ))
    run.event("error_budget_verified", role_rows=int(roles["rows"].sum()),
              horizon_rows=int(horizons["rows"].sum()))

## Why the broader interaction model is not an add-only ablation

Both challengers retained 64 features, but their feature sets differ. Re-ranking the broader bank displaced landing features. The score therefore combines the effect of adding interaction signals **and removing existing signals**. It does not establish that player interactions are harmful.

The displayed coefficients act on training-standardized features in the canonical field orientation. They are signed conditional weights, not causal effects or reliable standalone importance scores when predictors are correlated.

In [ ]:
coefficients = pd.DataFrame(selection["coefficient_rows"])
display(coefficients.round(4))
display(pd.DataFrame({
    "Removed landing feature": selection["removed_from_landing"],
    "Added interaction feature": selection["added_by_interaction"],
}))
display(Markdown(
    f"**Finding:** only {selection['overlap_count']} of the 64 columns are shared. "
    "`fraction__lateral_speed` was displaced, despite having a large standardized "
    "lateral coefficient in the winning model. That motivates a targeted add-back test; "
    "the coefficient alone does not prove the cause of the score difference."
))

## Next modeling decision

**Preserve the landing champion and run a genuinely additive interaction ablation.** Fit a correction to its remaining error, retaining the complete landing representation. Compare no correction, a small family-balanced interaction correction, and a role-conditioned correction. Fit screening, scaling, and hyperparameter choices inside chronological training folds; use the existing development validation only for the final comparison and leave the holdout untouched.

Prioritize receiver/defender relative motion, closest approach, and role × forecast-time behavior rather than blindly enlarging the bank. Report paired game-level RMSE differences, defense/horizon slices, feature stability, and inference cost. A nonlinear residual learner is a subsequent challenger, not something these ridge results have already validated.

**External research, not reproduced here:** the competition's third-place writeup describes pre-training and fine-tuning with a small trusted feature set. It supports investigating representation and training strategy—not treating thousands of columns as a performance guarantee. [Primary writeup](https://www.kaggle.com/competitions/nfl-big-data-bowl-2026-prediction/writeups/3rd-place-solution).

**Completion boundary:** the additive/role-conditioned experiments and final holdout evaluation have not yet run. The measured champion remains the one in the table above.

## Generate and download your own inference artifact

This competition uses the organizer's inference server, not a hand-built hidden-test CSV. In your existing SageMaker workspace, set `GENERATE_EXPORT = True` and run the next cell. It selects the measured winner from your completed local feature experiment, checks its source/model/split provenance, and creates `artifacts/kaggle/submission.ipynb`. The download link appears only after that succeeds.

You then open the generated notebook in Kaggle, attach the competition input, and run its local gateway yourself. The generated notebook also offers a link to `submission.parquet` when the local gateway creates it. Those sample predictions are not a hidden-test score. Any actual submission is a separate action you control. This project never calls a submit endpoint.

Keep this switch off in the public research notebook. Training and export switches are independent. Nothing is generated by merely reviewing or executing the default public notebook.

In [ ]:
import base64
import subprocess
import sys

from IPython.display import HTML

GENERATE_EXPORT = False
if GENERATE_EXPORT:
    local_summary = ROOT / "artifacts/features/summary.json"
    if not local_summary.is_file():
        raise FileNotFoundError("Run your local feature experiment before requesting an export.")
    requested_model = json.loads(local_summary.read_text())["selected_model"]
    subprocess.run(
        [sys.executable, str(ROOT / "kaggle/export.py"), "--model", requested_model],
        cwd=ROOT, check=True,
    )
    export_path = ROOT / "artifacts/kaggle/submission.ipynb"
    payload = base64.b64encode(export_path.read_bytes()).decode("ascii")
    display(HTML(
        '<a download="submission.ipynb" href="data:application/x-ipynb+json;base64,'
        + payload + '">Download the inference notebook you generated</a>'
    ))
else:
    display(Markdown(
        "**Export is off.** Set `GENERATE_EXPORT = True` "
        "to build and download your own artifact."
    ))